# Gold Layer - Instacart Analytics

This notebook performs business analytics on Silver layer data and creates aggregated Gold layer tables/views in the `big_data.gold` schema.

## Analysis Categories:
* **Sales Analytics**: Sales by time period, hour, reorder analysis by department/aisle
* **Customer Analytics**: Customer segmentation, purchase frequency, days between orders
* **Product Analytics**: Top products, product performance, department metrics

## Gold Layer Tables:
* `sales_by_time_period` - Sales metrics by period of day
* `sales_by_hour` - Hourly sales patterns
* `reorder_analysis_by_department` - Reorder rates by department
* `reorder_analysis_by_aisle` - Reorder rates by aisle
* `customer_segmentation` - Customer loyalty segments
* `customer_purchase_frequency` - Days between orders distribution
* `product_performance` - Top products with metrics
* `department_performance` - Department-level metrics

In [0]:
# PySpark functions
from pyspark.sql.functions import (
    col, count, countDistinct, avg, sum as _sum, round as _round, 
    max as _max, min as _min, when, percentile_approx
)

In [0]:
# Schema configuration
silver_schema = "big_data.silver"
gold_schema = "big_data.gold"

# Silver tables to analyze
silver_tables = ['orders', 'products_enriched', 'order_products']

print(f"Source schema: {silver_schema}")
print(f"Target schema: {gold_schema}")
print(f"Tables to analyze: {len(silver_tables)}")

In [0]:
# Display schemas and sample data for all Silver tables
print("Displaying Silver Layer Table Schemas and Samples\n")

for table_name in silver_tables:
    full_table_name = f"{silver_schema}.{table_name}"
    print(f"Table: {full_table_name}")
    
    # Show schema
    schema_df = spark.sql(f"DESCRIBE {full_table_name}")
    display(schema_df)
    
    # Show sample data
    print(f"Sample data (first 3 rows):")
    display(spark.table(full_table_name).limit(3))

## Sales Analysis by Time Period and Hour

In [0]:
orders = spark.table("big_data.silver.orders")
order_products = spark.table("big_data.silver.order_products")

print("Sales Analysis by Time Period\n")

sales_by_period = orders.join(order_products, "order_id") \
    .groupBy("period_of_day") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        count("product_id").alias("total_items_sold")
    )

total_orders = orders.select("order_id").distinct().count()

sales_by_period = sales_by_period.withColumn(
    "percentage_orders",
    _round((col("total_orders") / total_orders) * 100, 2)
).orderBy(col("total_orders").desc())

print("Sales by Period of Day:")
display(sales_by_period)

sales_by_hour = orders.join(order_products, "order_id") \
    .groupBy("order_hour_of_day") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        count("product_id").alias("total_items_sold")
    ) \
    .withColumn(
        "percentage_orders",
        _round((col("total_orders") / total_orders) * 100, 2)
    ).orderBy("order_hour_of_day")

print("\nSales by Hour of Day:")
display(sales_by_hour)

## Reorder Analysis by Product Category (Department and Aisle)

In [0]:
# Load Silver tables
order_products = spark.table(f"{silver_schema}.order_products")
products_enriched = spark.table(f"{silver_schema}.products_enriched")

print("Reorder Analysis by Product Category\n")

# Join to get product categories
orders_with_categories = order_products.join(products_enriched, "product_id")

# Reorder analysis by department
reorders_by_dept = orders_with_categories.groupBy("department") \
    .agg(
        count("*").alias("total_items"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items"),
    ) \
    .withColumn("reorder_rate", _round((col("reordered_items") / col("total_items")) * 100, 2)) \
    .orderBy(col("reordered_items").desc())

print("Reorders by Department:")
display(reorders_by_dept)

# Reorder analysis by aisle (top 10)
reorders_by_aisle = orders_with_categories.groupBy("aisle") \
    .agg(
        count("*").alias("total_items"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items"),
    ) \
    .withColumn("reorder_rate", _round((col("reordered_items") / col("total_items")) * 100, 2)) \
    .orderBy(col("reordered_items").desc()) \
    .limit(10)

print("\nReorders by Aisle (Top 10):")
display(reorders_by_aisle)

## Analysis of Days Between Customer Orders

In [0]:
# Load orders table
orders = spark.table(f"{silver_schema}.orders")

print("Average Days Between Orders Analysis\n")

# Overall statistics (excluding first orders)
overall_stats = orders.filter(col("is_first_order") == False) \
    .agg(
        _round(avg("days_since_prior_order"), 2).alias("avg_days"),
        _round(_min("days_since_prior_order"), 2).alias("min_days"),
        _round(_max("days_since_prior_order"), 2).alias("max_days"),
        _round(percentile_approx("days_since_prior_order", 0.5), 2).alias("median_days"),
        count("*").alias("total_repeat_orders")
    )

print("Overall Statistics:")
display(overall_stats)

# Average days by user
user_avg_days = orders.filter(col("is_first_order") == False) \
    .groupBy("user_id") \
    .agg(
        _round(avg("days_since_prior_order"), 2).alias("avg_days_between_orders"),
        count("*").alias("repeat_order_count")
    ) \
    .orderBy(col("repeat_order_count").desc())

print("\nAverage Days Between Orders by User (Sample of 10):")
display(user_avg_days.limit(10))

# Distribution of days between orders
days_distribution = orders.filter(col("is_first_order") == False) \
    .selectExpr(
        "CASE \
            WHEN days_since_prior_order <= 7 THEN '1-7 days' \
            WHEN days_since_prior_order <= 14 THEN '8-14 days' \
            WHEN days_since_prior_order <= 21 THEN '15-21 days' \
            WHEN days_since_prior_order <= 30 THEN '22-30 days' \
            ELSE '30+ days' \
        END as days_range"
    ) \
    .groupBy("days_range") \
    .agg(count("*").alias("order_count")) \
    .orderBy(
        col("days_range").isin('1-7 days', '8-14 days', '15-21 days', '22-30 days', '30+ days').desc(),
        "days_range"
    )

print("\nDistribution of Days Between Orders:")
display(days_distribution)

## Customer Behavior Analysis and Segmentation Based on Orders

In [0]:
# Load Silver tables
orders = spark.table(f"{silver_schema}.orders")
order_products = spark.table(f"{silver_schema}.order_products")

print("Customer Relationship Analysis\n")

# Customer order frequency
customer_orders = orders.groupBy("user_id") \
    .agg(
        count("order_id").alias("total_orders"),
        _max("order_number").alias("max_order_number")
    )

# Customer segmentation by order frequency
order_frequency_dist = customer_orders.selectExpr(
    "CASE \
        WHEN total_orders = 1 THEN '1 order (New)' \
        WHEN total_orders BETWEEN 2 AND 5 THEN '2-5 orders (Occasional)' \
        WHEN total_orders BETWEEN 6 AND 10 THEN '6-10 orders (Regular)' \
        WHEN total_orders BETWEEN 11 AND 20 THEN '11-20 orders (Frequent)' \
        ELSE '20+ orders (Loyal)' \
    END as customer_segment"
) \
.groupBy("customer_segment") \
.agg(count("*").alias("customer_count")) \
.orderBy(
    col("customer_segment").isin('1 order (New)', '2-5 orders (Occasional)', '6-10 orders (Regular)', '11-20 orders (Frequent)', '20+ orders (Loyal)').desc(),
    "customer_segment"
)

print("Customer Order Frequency Distribution:")
display(order_frequency_dist)

# Top loyal customers
top_customers = orders.join(order_products, "order_id") \
    .groupBy("user_id") \
    .agg(
        count("order_id").alias("total_orders"),
        count("product_id").alias("total_items_purchased"),
        _round(avg("days_since_prior_order"), 2).alias("avg_days_between_orders")
    ) \
    .orderBy(col("total_orders").desc()) \
    .limit(10)

print("\nTop 10 Most Loyal Customers:")
display(top_customers)

# Customer lifecycle metrics
lifecycle_metrics = orders.agg(
    count("user_id").alias("total_orders"),
    _round(avg("order_number"), 2).alias("avg_orders_per_customer"),
    _sum(when(col("is_first_order") == True, 1).otherwise(0)).alias("first_time_orders"),
    _sum(when(col("is_first_order") == False, 1).otherwise(0)).alias("repeat_orders")
) \
.withColumn("repeat_order_rate", _round((col("repeat_orders") / col("total_orders")) * 100, 2))

print("\nCustomer Lifecycle Metrics:")
display(lifecycle_metrics)

# Average basket size by customer segment
customer_basket = orders.join(order_products, "order_id") \
    .groupBy("user_id", "order_id") \
    .agg(count("product_id").alias("items_in_basket"))

avg_basket_by_orders = customer_orders.join(customer_basket, "user_id") \
    .selectExpr(
        "CASE \
            WHEN total_orders = 1 THEN 'New (1 order)' \
            WHEN total_orders BETWEEN 2 AND 5 THEN 'Occasional (2-5)' \
            WHEN total_orders BETWEEN 6 AND 10 THEN 'Regular (6-10)' \
            WHEN total_orders BETWEEN 11 AND 20 THEN 'Frequent (11-20)' \
            ELSE 'Loyal (20+)' \
        END as segment",
        "items_in_basket"
    ) \
    .groupBy("segment") \
    .agg(
        _round(avg("items_in_basket"), 2).alias("avg_basket_size")
    ) \
    .orderBy(
        col("segment").isin('New (1 order)', 'Occasional (2-5)', 'Regular (6-10)', 'Frequent (11-20)', 'Loyal (20+)').desc(),
        "segment"
    )

print("\nAverage Basket Size by Customer Segment:")
display(avg_basket_by_orders)

## Product and Category Performance Analysis in Sales

In [0]:
# Load Silver tables
order_products = spark.table(f"{silver_schema}.order_products")
products_enriched = spark.table(f"{silver_schema}.products_enriched")
orders = spark.table(f"{silver_schema}.orders")

print("Product Performance Analysis\n")

# Join to get product details
orders_with_products = order_products.join(products_enriched, "product_id")

# Top 10 most popular products
top_products = orders_with_products.groupBy("product_id", "product_name", "department", "aisle") \
    .agg(
        count("*").alias("times_ordered"),
        countDistinct("order_id").alias("unique_orders"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("times_reordered")
    ) \
    .withColumn("reorder_rate", _round((col("times_reordered") / col("times_ordered")) * 100, 2)) \
    .orderBy(col("times_ordered").desc()) \
    .limit(10)

print("Top 10 Most Popular Products:")
display(top_products)

# Top 10 products by reorder rate
top_reorder_products = orders_with_products.groupBy("product_id", "product_name", "department", "aisle") \
    .agg(
        count("*").alias("times_ordered"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("times_reordered")
    ) \
    .filter(col("times_ordered") >= 100) \
    .withColumn("reorder_rate", _round((col("times_reordered") / col("times_ordered")) * 100, 2)) \
    .orderBy(col("reorder_rate").desc()) \
    .limit(10)

print("\nTop 10 Products by Reorder Rate (min 100 orders):")
display(top_reorder_products)

# Department performance summary
dept_performance = orders_with_products.groupBy("department") \
    .agg(
        countDistinct("product_id").alias("unique_products"),
        count("*").alias("total_items_sold"),
        countDistinct("order_id").alias("unique_orders"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items")
    ) \
    .withColumn("reorder_rate", _round((col("reordered_items") / col("total_items_sold")) * 100, 2)) \
    .withColumn("avg_items_per_order", _round(col("total_items_sold") / col("unique_orders"), 2)) \
    .orderBy(col("total_items_sold").desc())

print("\nDepartment Performance Summary:")
display(dept_performance)

# Add-to-cart order analysis
add_to_cart_analysis = orders_with_products.groupBy("add_to_cart_order") \
    .agg(
        count("*").alias("product_count"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_count")
    ) \
    .withColumn("reorder_rate", _round((col("reordered_count") / col("product_count")) * 100, 2)) \
    .filter(col("add_to_cart_order") <= 10) \
    .orderBy("add_to_cart_order")

print("\nProduct Add-to-Cart Order Analysis:")
display(add_to_cart_analysis)

# Cross-category purchase patterns
order_dept_diversity = orders_with_products.groupBy("order_id") \
    .agg(countDistinct("department").alias("dept_count"))

diversity_distribution = order_dept_diversity.groupBy("dept_count") \
    .agg(count("*").alias("order_count")) \
    .orderBy("dept_count")

print("\nCross-Category Purchase Patterns:")
display(diversity_distribution)

## Create Gold Layer Tables and Views with Sales, Customer, and Product Metrics

In [0]:
print("Creating Gold Layer Tables\n")

# Create gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"Schema {gold_schema} ready\n")

# Load Silver tables
orders = spark.table(f"{silver_schema}.orders")
order_products = spark.table(f"{silver_schema}.order_products")
products_enriched = spark.table(f"{silver_schema}.products_enriched")

# 1. Sales by time period
print("Creating sales_by_time_period...")
sales_by_period = orders.join(order_products, "order_id") \
    .groupBy("period_of_day") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        count("product_id").alias("total_items_sold")
    )
total_orders = orders.select("order_id").distinct().count()
sales_by_period = sales_by_period.withColumn(
    "percentage_orders",
    _round((col("total_orders") / total_orders) * 100, 2)
)
sales_by_period.write.mode("overwrite").saveAsTable(f"{gold_schema}.sales_by_time_period")
print("Created")

# 2. Sales by hour
print("Creating sales_by_hour...")
sales_by_hour = orders.join(order_products, "order_id") \
    .groupBy("order_hour_of_day") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        count("product_id").alias("total_items_sold")
    ) \
    .withColumn("percentage_orders", _round((col("total_orders") / total_orders) * 100, 2))
sales_by_hour.write.mode("overwrite").saveAsTable(f"{gold_schema}.sales_by_hour")
print("Created")

# 3. Reorder analysis by department
print("Creating reorder_analysis_by_department...")
orders_with_categories = order_products.join(products_enriched, "product_id")
reorders_by_dept = orders_with_categories.groupBy("department") \
    .agg(
        count("*").alias("total_items"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items")
    ) \
    .withColumn("reorder_rate", _round((col("reordered_items") / col("total_items")) * 100, 2))
reorders_by_dept.write.mode("overwrite").saveAsTable(f"{gold_schema}.reorder_analysis_by_department")
print("Created")

# 4. Reorder analysis by aisle
print("Creating reorder_analysis_by_aisle...")
reorders_by_aisle = orders_with_categories.groupBy("aisle") \
    .agg(
        count("*").alias("total_items"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items")
    ) \
    .withColumn("reorder_rate", _round((col("reordered_items") / col("total_items")) * 100, 2))
reorders_by_aisle.write.mode("overwrite").saveAsTable(f"{gold_schema}.reorder_analysis_by_aisle")
print("Created")

# 5. Customer purchase frequency
print("Creating customer_purchase_frequency...")
days_distribution = orders.filter(col("is_first_order") == False) \
    .selectExpr(
        "CASE \
            WHEN days_since_prior_order <= 7 THEN '1-7 days' \
            WHEN days_since_prior_order <= 14 THEN '8-14 days' \
            WHEN days_since_prior_order <= 21 THEN '15-21 days' \
            WHEN days_since_prior_order <= 30 THEN '22-30 days' \
            ELSE '30+ days' \
        END as days_range"
    ) \
    .groupBy("days_range") \
    .agg(count("*").alias("order_count"))
days_distribution.write.mode("overwrite").saveAsTable(f"{gold_schema}.customer_purchase_frequency")
print("Created")

# 6. Customer segmentation
print("Creating customer_segmentation...")
customer_orders = orders.groupBy("user_id") \
    .agg(
        count("order_id").alias("total_orders"),
        _max("order_number").alias("max_order_number")
    )
customer_segments = customer_orders.selectExpr(
    "user_id",
    "total_orders",
    "CASE \
        WHEN total_orders = 1 THEN 'New' \
        WHEN total_orders BETWEEN 2 AND 5 THEN 'Occasional' \
        WHEN total_orders BETWEEN 6 AND 10 THEN 'Regular' \
        WHEN total_orders BETWEEN 11 AND 20 THEN 'Frequent' \
        ELSE 'Loyal' \
    END as customer_segment"
)
customer_segments.write.mode("overwrite").saveAsTable(f"{gold_schema}.customer_segmentation")
print("Created")

# 7. Product performance
print("Creating product_performance...")
top_products = orders_with_categories.groupBy("product_id", "product_name", "department", "aisle") \
    .agg(
        count("*").alias("times_ordered"),
        countDistinct("order_id").alias("unique_orders"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("times_reordered")
    ) \
    .withColumn("reorder_rate", _round((col("times_reordered") / col("times_ordered")) * 100, 2))
top_products.write.mode("overwrite").saveAsTable(f"{gold_schema}.product_performance")
print("Created")

# 8. Department performance
print("Creating department_performance...")
dept_performance = orders_with_categories.groupBy("department") \
    .agg(
        countDistinct("product_id").alias("unique_products"),
        count("*").alias("total_items_sold"),
        countDistinct("order_id").alias("unique_orders"),
        _sum(when(col("reordered") == True, 1).otherwise(0)).alias("reordered_items")
    ) \
    .withColumn("reorder_rate", _round((col("reordered_items") / col("total_items_sold")) * 100, 2)) \
    .withColumn("avg_items_per_order", _round(col("total_items_sold") / col("unique_orders"), 2))
dept_performance.write.mode("overwrite").saveAsTable(f"{gold_schema}.department_performance")
print("Created")

print(f"\nAll Gold layer tables created successfully in {gold_schema}!")